In [ ]:
import os
import requests
import time

# 1. Creamos la estructura de carpetas que exige load_files
os.makedirs('IB/Talleres/txt-files/libros', exist_ok=True)

print("Iniciando descarga de 100 libros...")
libros_descargados = 0
id_libro = 1

# 2. Bucle que se detiene al llegar a 100
while libros_descargados < 100:
    # URL directa al archivo de texto plano en Gutenberg
    url = f"https://www.gutenberg.org/cache/epub/{id_libro}/pg{id_libro}.txt"
    respuesta = requests.get(url)
    
    # Si el libro existe (código 200), lo guardamos
    if respuesta.status_code == 200:
        ruta = f"IB/Talleres/txt-files/libros/libro_{id_libro}.txt"
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write(respuesta.text)
            
        libros_descargados += 1
        print(f"Descargado: libro_{id_libro}.txt ({libros_descargados}/100)")
        
        # Pausa de 1 segundo para que el servidor de Gutenberg no te bloquee la IP
        time.sleep(1)
        
    id_libro += 1

print("¡Descarga completada! Ya puedes usar load_files('mi_carpeta_gutenberg')")

In [ ]:
import math
import string
from sklearn.datasets import load_files

# Funciones: 
def obtener_matriz_tf(vocabulario, documentos):
    matriz_tf = []
    for doc in documentos:
        fila_tf = []
        for termino in vocabulario:
            frecuencia = doc.count(termino)
            fila_tf.append(frecuencia)
        matriz_tf.append(fila_tf)
    return matriz_tf

def obtener_df(vocabulario, matriz_tf):
    lista_df = []
    num_documentos = len(matriz_tf)
    for j in range(len(vocabulario)):
        contador_docs = 0
        for i in range(num_documentos):
            if matriz_tf[i][j] > 0:
                contador_docs += 1
        lista_df.append(contador_docs)
    return lista_df

def calcular_matriz_tfidf(matriz_tf, lista_df):
    matriz_tfidf = []
    num_documentos = len(matriz_tf)
    num_terminos = len(lista_df)
    for i in range(num_documentos):
        fila_tfidf = []
        for j in range(num_terminos):
            tf = matriz_tf[i][j]
            df_termino = lista_df[j]
            if df_termino > 0:
                idf = math.log10(num_documentos / df_termino)
                tfidf = tf * idf
            else:
                tfidf = 0.0
            fila_tfidf.append(round(tfidf, 4))
        matriz_tfidf.append(fila_tfidf)
    return matriz_tfidf

In [ ]:
# Carga de archivos y procesamiento
if __name__ == "__main__":
    # Carga de la carpeta con archivos .txt
    corpus = load_files('/Users/brandonfreire/Documents/Proyectos/Notebooks-RI/IB/Talleres/txt-files', encoding='utf-8')
    
    # Tomamos todos los documentos crudos del corpus
    textos_crudos = corpus.data[:]
    print(f"Se cargaron {len(textos_crudos)} documentos crudos.")

    # Listas para guardar los datos procesados
    documentos_procesados = []
    vocabulario_set = set() # Usamos un set para evitar palabras duplicadas rápidamente
    
    # Stopwords básicas en inglés (asumiendo que los libros de Gutenberg están en inglés)
    stopwords = {"the", "and", "to", "of", "a", "i", "in", "it", "that", "was", "he", "is", "for", "as", "on", "with"}

    # Limpiamos cada texto
    for i, texto in enumerate(textos_crudos):
        #texto_recortado = texto[:2000] 
        
        # 1. Minúsculas
        texto_limpio = texto.lower()
        
        # 2. Quitar puntuación
        texto_limpio = texto_limpio.translate(str.maketrans('', '', string.punctuation))
        
        # 3. Dividir en lista de palabras (tokenizar)
        palabras = texto_limpio.split()
        
        # 4. Filtrar stopwords y números
        palabras_filtradas = [p for p in palabras if p not in stopwords and p.isalpha()]
        
        documentos_procesados.append(palabras_filtradas)
        vocabulario_set.update(palabras_filtradas)
        
        #print(f"Documento procesado: {i+1} de 100.")

    # Convertimos el set a una lista ordenada alfabéticamente
    vocabulario_final = sorted(list(vocabulario_set))
    print(f"El vocabulario final tiene {len(vocabulario_final)} términos únicos.")
    
    # Uso de las funciones
    print("\nCalculando Matriz TF...")
    matriz_tf_100 = obtener_matriz_tf(vocabulario_final, documentos_procesados)
    
    print("Calculando Vector DF...")
    lista_df_100 = obtener_df(vocabulario_final, matriz_tf_100)
    
    print("Calculando Matriz TF-IDF final...")
    matriz_tfidf_100 = calcular_matriz_tfidf(matriz_tf_100, lista_df_100)
    
    print("\n¡Cálculo terminado con éxito!")